# BFCL_v3 Tool Extraction Analysis

This notebook analyzes the tool definitions extracted from BFCL_v3 dataset.

**Data Sources:**
- `bfcl_v3_all_tool_definitions.jsonl` - All 1,674 tool definitions
- `bfcl_v3_invocation_examples.jsonl` - 3,641 invocation examples

In [1]:
import json
from collections import defaultdict, Counter
from pathlib import Path
import pandas as pd
from pprint import pprint

## 1. Load Data

In [2]:
# Load all tool definitions
tools = []
with open('bfcl_v3_all_tool_definitions.jsonl', 'r') as f:
    for line in f:
        tools.append(json.loads(line))

print(f"✅ Loaded {len(tools)} tool definitions")

✅ Loaded 1674 tool definitions


In [3]:
# Load invocation examples
examples = []
with open('bfcl_v3_invocation_examples.jsonl', 'r') as f:
    for line in f:
        examples.append(json.loads(line))

print(f"✅ Loaded {len(examples)} invocation examples")

✅ Loaded 3641 invocation examples


## 2. Overall Statistics

In [4]:
print("=" * 80)
print("OVERALL TOOL STATISTICS")
print("=" * 80)
print(f"\n📊 Total unique tools: {len(tools)}")
print(f"📊 Total invocation examples: {len(examples)}")

# Count by source
source_counts = Counter(t['source'] for t in tools)
print(f"\n📁 Tools by source ({len(source_counts)} sources):")
for source, count in source_counts.most_common():
    pct = count / len(tools) * 100
    print(f"  {source:40} {count:4} tools ({pct:5.1f}%)")

# Parameter statistics
with_params = sum(1 for t in tools if t.get('parameters'))
with_required = sum(1 for t in tools if t.get('parameters', {}).get('required'))
with_properties = sum(1 for t in tools if t.get('parameters', {}).get('properties'))

print(f"\n📋 Parameter statistics:")
print(f"  With parameters field:    {with_params:4} ({with_params/len(tools)*100:5.1f}%)")
print(f"  With required params:     {with_required:4} ({with_required/len(tools)*100:5.1f}%)")
print(f"  With properties:          {with_properties:4} ({with_properties/len(tools)*100:5.1f}%)")

# Example statistics
unique_functions_in_examples = len(set(e['function_name'] for e in examples))
print(f"\n📊 Example statistics:")
print(f"  Unique functions in examples: {unique_functions_in_examples}")
print(f"  Total invocation instances:   {len(examples)}")

OVERALL TOOL STATISTICS

📊 Total unique tools: 1674
📊 Total invocation examples: 3641

📁 Tools by source (14 sources):
  BFCL_v3_live_multiple.json                420 tools ( 25.1%)
  BFCL_v3_simple.json                       368 tools ( 22.0%)
  BFCL_v3_multiple.json                     219 tools ( 13.1%)
  BFCL_v3_live_irrelevance.json             206 tools ( 12.3%)
  multi_turn_func_doc                       129 tools (  7.7%)
  BFCL_v3_parallel_multiple.json            127 tools (  7.6%)
  BFCL_v3_live_simple.json                   81 tools (  4.8%)
  BFCL_v3_parallel.json                      52 tools (  3.1%)
  BFCL_v3_exec_simple.json                   34 tools (  2.0%)
  BFCL_v3_exec_multiple.json                 20 tools (  1.2%)
  BFCL_v3_live_parallel_multiple.json        10 tools (  0.6%)
  BFCL_v3_live_relevance.json                 4 tools (  0.2%)
  BFCL_v3_live_parallel.json                  3 tools (  0.2%)
  BFCL_v3_exec_parallel_multiple.json         1 tools (  0.1%)

## 3. Tool Groups by Source

In [5]:
# Group tools by source
tools_by_source = defaultdict(list)
for tool in tools:
    tools_by_source[tool['source']].append(tool)

print("=" * 80)
print("TOOL GROUPS BY SOURCE")
print("=" * 80)

for source in sorted(tools_by_source.keys()):
    tool_list = tools_by_source[source]
    print(f"\n📁 {source} ({len(tool_list)} tools)")
    print("-" * 80)
    
    # Show first 10 tools
    for i, tool in enumerate(tool_list[:10], 1):
        desc = tool.get('api_description', '')[:60]
        print(f"  {i:2}. {tool['api_name']:40} - {desc}...")
    
    if len(tool_list) > 10:
        print(f"  ... and {len(tool_list) - 10} more")

TOOL GROUPS BY SOURCE

📁 BFCL_v3_exec_multiple.json (20 tools)
--------------------------------------------------------------------------------
   1. convert_binary_to_decimal                - Converts a binary number to a decimal number....
   2. convert_decimal_to_hex                   - Converts a decimal number to a hexadecimal number....
   3. calculate_slope                          - Calculates the slope of the linear regression line from a se...
   4. calculate_intercept                      - Calculates the y-intercept of the linear regression line fro...
   5. predict_value                            - Predicts the value of y given the slope, intercept, and an x...
   6. inflation_adjustment                     - Adjusts an amount for inflation....
   7. adjust_for_inflation                     - Adjusts the investment value for inflation for each year....
   8. calculate_basal_metabolic_rate           - Calculates the Basal Metabolic Rate (BMR) of a person....
   9. calculat

## 4. Multi-turn Tool Categories

In [6]:
# Analyze multi-turn categories
multi_turn_tools = [t for t in tools if t['source'] == 'multi_turn_func_doc']

print("=" * 80)
print("MULTI-TURN TOOL CATEGORIES")
print("=" * 80)

# Group by category
tools_by_category = defaultdict(list)
for tool in multi_turn_tools:
    cat = tool.get('category', 'Unknown')
    tools_by_category[cat].append(tool)

print(f"\n📊 Categories: {len(tools_by_category)}")
print(f"📊 Tools per category:")

for category in sorted(tools_by_category.keys()):
    tool_list = tools_by_category[category]
    print(f"\n  {category:20} ({len(tool_list):2} tools)")
    for tool in tool_list[:5]:
        print(f"    - {tool['api_name']}")

MULTI-TURN TOOL CATEGORIES

📊 Categories: 8
📊 Tools per category:

  Communication        (10 tools)
    - add_contact
    - delete_message
    - get_message_stats
    - get_user_id
    - list_users

  Events               ( 9 tools)
    - close_ticket
    - create_ticket
    - edit_ticket
    - get_ticket
    - get_user_tickets

  Finance              (22 tools)
    - add_to_watchlist
    - cancel_order
    - filter_stocks_by_price
    - fund_account
    - get_account_info

  Posting Api          (14 tools)
    - authenticate_twitter
    - comment
    - follow_user
    - get_tweet
    - get_tweet_comments

  Science              (17 tools)
    - absolute_value
    - add
    - divide
    - imperial_si_conversion
    - logarithm

  Storage              (18 tools)
    - cat
    - cd
    - cp
    - diff
    - du

  Travel Booking       (17 tools)
    - authenticate_travel
    - book_flight
    - cancel_booking
    - compute_exchange_rate
    - contact_customer_support

  Vehicle Control  

## 5. Top Tools with Examples

For each tool group, show top-5 tools with example inputs-outputs.

In [7]:
# Create lookup for examples by function name
examples_by_function = defaultdict(list)
for ex in examples:
    examples_by_function[ex['function_name']].append(ex)

print(f"📊 Examples indexed for {len(examples_by_function)} unique functions")

📊 Examples indexed for 86 unique functions


In [8]:
def display_tool_with_examples(tool, examples_by_function, max_examples=3):
    """Display a tool definition with example invocations."""
    
    api_name = tool['api_name']
    print(f"\n{'=' * 80}")
    print(f"🔧 Tool: {api_name}")
    print(f"{'=' * 80}")
    
    # Description
    desc = tool.get('api_description', 'No description')
    print(f"\n📝 Description: {desc}")
    
    # Parameters
    params = tool.get('parameters', {})
    if params:
        print(f"\n📋 Parameters:")
        print(f"  Type: {params.get('type', 'N/A')}")
        
        required = params.get('required', [])
        if required:
            print(f"  Required: {', '.join(required)}")
        
        properties = params.get('properties', {})
        if properties:
            print(f"\n  Properties:")
            for param_name, param_info in list(properties.items())[:5]:
                param_type = param_info.get('type', 'unknown')
                param_desc = param_info.get('description', '')[:50]
                print(f"    - {param_name:20} ({param_type:10}): {param_desc}")
    
    # Examples
    function_examples = examples_by_function.get(api_name, [])
    
    if function_examples:
        print(f"\n💡 Example Invocations ({len(function_examples)} total):")
        
        for i, ex in enumerate(function_examples[:max_examples], 1):
            print(f"\n  Example {i}:")
            print(f"    User: {ex.get('user_message', 'N/A')[:100]}...")
            print(f"    Call: {ex.get('call_string', 'N/A')}")
            print(f"    Args: {json.dumps(ex.get('arguments', {}), indent=6)}")
            
            # Show initial config if available
            if 'initial_config' in ex:
                config_keys = list(ex['initial_config'].keys())[:3]
                print(f"    Config: {', '.join(config_keys)}...")
    else:
        print(f"\n⚠️  No example invocations found")
    
    print()

### 5.1 Top Tools from Multi-turn Categories

In [9]:
# Show top-5 tools from each multi-turn category
print("\n" + "=" * 80)
print("TOP TOOLS BY MULTI-TURN CATEGORY")
print("=" * 80)

for category in sorted(tools_by_category.keys()):
    tool_list = tools_by_category[category]
    
    print(f"\n\n{'#' * 80}")
    print(f"# Category: {category} ({len(tool_list)} tools)")
    print(f"{'#' * 80}")
    
    # Show top 5 tools (or fewer if less available)
    for tool in tool_list[:5]:
        display_tool_with_examples(tool, examples_by_function, max_examples=2)


TOP TOOLS BY MULTI-TURN CATEGORY


################################################################################
# Category: Communication (10 tools)
################################################################################

🔧 Tool: add_contact

📝 Description: This tool belongs to the Message API, which is used to manage user interactions in a workspace. Tool description: Add a contact to the workspace.

📋 Parameters:
  Type: dict
  Required: user_name

  Properties:
    - user_name            (string    ): User name of contact to be added.

💡 Example Invocations (6 total):

  Example 1:
    User: Logging in as USR001. Lastly, upon completion of our file review, kindly message my colleague, John ...
    Call: add_contact(user_name='John Levy')
    Args: {
      "user_name": "John Levy"
}
    Config: GorillaFileSystem...

  Example 2:
    User: Please dispatch of the report to Kelly, I need to add her contact (Kelly), in the format of 'Kelly T...
    Call: add_contact(user_na

### 5.2 Top Tools from Live APIs

In [10]:
# Show top-5 tools from live API sources
print("\n" + "=" * 80)
print("TOP TOOLS FROM LIVE APIs")
print("=" * 80)

live_sources = [s for s in tools_by_source.keys() if 'live' in s]

# Get top 5 most used live tools (by number of occurrences)
live_tools = []
for source in live_sources:
    live_tools.extend(tools_by_source[source])

# Show first 5 live tools
for i, tool in enumerate(live_tools[:5], 1):
    print(f"\n\n{'='*80}")
    print(f"Live Tool {i}")
    print(f"{'='*80}")
    display_tool_with_examples(tool, examples_by_function, max_examples=1)


TOP TOOLS FROM LIVE APIs


Live Tool 1

🔧 Tool: get_user_info

📝 Description: Retrieve details for a specific user by their unique identifier.

📋 Parameters:
  Type: dict
  Required: user_id

  Properties:
    - user_id              (integer   ): The unique identifier of the user. It is used to f
    - special              (string    ): Any special information or parameters that need to

⚠️  No example invocations found



Live Tool 2

🔧 Tool: github_star

📝 Description: Generates a URL for tracking the star history of specified GitHub repositories, with the option to align them on the same timeline.

📋 Parameters:
  Type: dict
  Required: repos

  Properties:
    - repos                (string    ): A comma-separated list of GitHub repositories to t
    - aligned              (boolean   ): Whether to align the repositories on the same time

⚠️  No example invocations found



Live Tool 3

🔧 Tool: uber.ride

📝 Description: Finds a suitable Uber ride for customers based on their locati

### 5.3 Top Tools from Simple Tests

In [11]:
# Show top-5 tools from simple test source
print("\n" + "=" * 80)
print("TOP TOOLS FROM SIMPLE TESTS")
print("=" * 80)

simple_tools = tools_by_source.get('BFCL_v3_simple.json', [])

for i, tool in enumerate(simple_tools[:5], 1):
    print(f"\n\n{'='*80}")
    print(f"Simple Tool {i}")
    print(f"{'='*80}")
    display_tool_with_examples(tool, examples_by_function, max_examples=1)


TOP TOOLS FROM SIMPLE TESTS


Simple Tool 1

🔧 Tool: calculate_triangle_area

📝 Description: Calculate the area of a triangle given its base and height.

📋 Parameters:
  Type: dict
  Required: base, height

  Properties:
    - base                 (integer   ): The base of the triangle.
    - height               (integer   ): The height of the triangle.
    - unit                 (string    ): The unit of measure (defaults to 'units' if not sp

⚠️  No example invocations found



Simple Tool 2

🔧 Tool: math.factorial

📝 Description: Calculate the factorial of a given number.

📋 Parameters:
  Type: dict
  Required: number

  Properties:
    - number               (integer   ): The number for which factorial needs to be calcula

⚠️  No example invocations found



Simple Tool 3

🔧 Tool: math.hypot

📝 Description: Calculate the Euclidean norm, sqrt(sum(squares)), the length of the vector from the origin to point (x, y) which is the hypotenuse of the right triangle.

📋 Parameters:
  Type

## 6. Parameter Analysis

In [12]:
# Analyze parameter types
print("=" * 80)
print("PARAMETER TYPE ANALYSIS")
print("=" * 80)

param_types = Counter()
required_counts = []
optional_counts = []

for tool in tools:
    params = tool.get('parameters', {})
    if params:
        # Count parameter types
        properties = params.get('properties', {})
        for prop_info in properties.values():
            param_type = prop_info.get('type', 'unknown')
            param_types[param_type] += 1
        
        # Count required vs optional
        required = params.get('required', [])
        required_counts.append(len(required))
        
        optional = [p for p in properties.keys() if p not in required]
        optional_counts.append(len(optional))

print(f"\n📊 Parameter type distribution:")
for ptype, count in param_types.most_common():
    print(f"  {ptype:15} {count:5} occurrences")

if required_counts:
    print(f"\n📊 Required parameters per tool:")
    print(f"  Min:    {min(required_counts)}")
    print(f"  Max:    {max(required_counts)}")
    print(f"  Mean:   {sum(required_counts)/len(required_counts):.2f}")
    print(f"  Median: {sorted(required_counts)[len(required_counts)//2]}")

if optional_counts:
    print(f"\n📊 Optional parameters per tool:")
    print(f"  Min:    {min(optional_counts)}")
    print(f"  Max:    {max(optional_counts)}")
    print(f"  Mean:   {sum(optional_counts)/len(optional_counts):.2f}")

PARAMETER TYPE ANALYSIS

📊 Parameter type distribution:
  string           2562 occurrences
  integer           926 occurrences
  float             363 occurrences
  boolean           347 occurrences
  array             284 occurrences
  dict               48 occurrences
  any                 7 occurrences
  tuple               6 occurrences

📊 Required parameters per tool:
  Min:    0
  Max:    9
  Mean:   1.79
  Median: 2

📊 Optional parameters per tool:
  Min:    0
  Max:    27
  Mean:   0.93


## 7. Example Coverage Analysis

In [13]:
# Analyze which tools have examples
print("=" * 80)
print("EXAMPLE COVERAGE ANALYSIS")
print("=" * 80)

tools_with_examples = 0
tools_without_examples = 0
example_counts = []

for tool in tools:
    api_name = tool['api_name']
    count = len(examples_by_function.get(api_name, []))
    if count > 0:
        tools_with_examples += 1
        example_counts.append(count)
    else:
        tools_without_examples += 1

print(f"\n📊 Coverage:")
print(f"  Tools with examples:    {tools_with_examples:4} ({tools_with_examples/len(tools)*100:5.1f}%)")
print(f"  Tools without examples: {tools_without_examples:4} ({tools_without_examples/len(tools)*100:5.1f}%)")

if example_counts:
    print(f"\n📊 Examples per tool (for tools with examples):")
    print(f"  Min:  {min(example_counts)}")
    print(f"  Max:  {max(example_counts)}")
    print(f"  Mean: {sum(example_counts)/len(example_counts):.2f}")

# Top 10 tools by number of examples
print(f"\n📊 Top 10 tools by number of examples:")
sorted_by_examples = sorted(
    [(t['api_name'], len(examples_by_function.get(t['api_name'], []))) for t in tools],
    key=lambda x: -x[1]
)
for i, (name, count) in enumerate(sorted_by_examples[:10], 1):
    print(f"  {i:2}. {name:40} {count:4} examples")

EXAMPLE COVERAGE ANALYSIS

📊 Coverage:
  Tools with examples:      85 (  5.1%)
  Tools without examples: 1589 ( 94.9%)

📊 Examples per tool (for tools with examples):
  Min:  2
  Max:  157
  Mean: 42.64

📊 Top 10 tools by number of examples:
   1. cd                                        157 examples
   2. startEngine                               132 examples
   3. get_stock_info                            129 examples
   4. lockDoors                                 126 examples
   5. book_flight                               123 examples
   6. get_flight_cost                           108 examples
   7. get_zipcode_based_on_city                 108 examples
   8. post_tweet                                103 examples
   9. get_order_details                          97 examples
  10. fillFuelTank                               96 examples


## 8. Summary Statistics

In [14]:
# Final summary
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

print(f"\n📊 Overall Statistics:")
print(f"  Total unique tools:              {len(tools):,}")
print(f"  Total invocation examples:       {len(examples):,}")
print(f"  Unique sources:                  {len(source_counts)}")
print(f"  Multi-turn categories:           {len(tools_by_category)}")
print(f"  Tools with examples:             {tools_with_examples:,} ({tools_with_examples/len(tools)*100:.1f}%)")

print(f"\n📊 Data Quality:")
print(f"  Tools with parameters:           {with_params:,} ({with_params/len(tools)*100:.1f}%)")
print(f"  Tools with required params:      {with_required:,} ({with_required/len(tools)*100:.1f}%)")
print(f"  Tools with property definitions: {with_properties:,} ({with_properties/len(tools)*100:.1f}%)")

print(f"\n📊 Diversity:")
print(f"  Parameter types:                 {len(param_types)}")
print(f"  Avg required params (when present): {sum(required_counts)/len(required_counts):.2f}" if required_counts else "N/A")

print(f"\n✅ Analysis complete!")


SUMMARY

📊 Overall Statistics:
  Total unique tools:              1,674
  Total invocation examples:       3,641
  Unique sources:                  14
  Multi-turn categories:           8
  Tools with examples:             85 (5.1%)

📊 Data Quality:
  Tools with parameters:           1,674 (100.0%)
  Tools with required params:      1,599 (95.5%)
  Tools with property definitions: 1,633 (97.6%)

📊 Diversity:
  Parameter types:                 8
  Avg required params (when present): 1.79

✅ Analysis complete!
